# Task 3: Occasion and Gender Classification


## 1. Introduction

This notebook addresses **Task 3: Occasion and Gender Classification** for Assignment 2 by predicting the catalogue gender category and intended occasion of a fashion product from its RGB image. The task is formulated as **two separate multi-class classification problems: one model predicts gender and another predicts occasion (the metadata field `usage`)**.

Three neural-network approaches are developed and trained from scratch using TensorFlow/Keras:

- **Shallow MLP (Baseline):** A fully connected network with one 256-unit hidden layer operating on flattened image pixels.
- **Deeper MLP:** A fully connected network with hidden layers of 256, 128, and 64 units, using dropout to reduce overfitting.
- **Convolutional Neural Network (CNN):** A spatial model that learns local image patterns through convolutional layers. The CNN family includes three declared architecture/scheduling configurations.

Performance is assessed using:

- **Accuracy:** The proportion of correctly classified images.
- **Macro-F1:** The primary selection metric, giving equal weight to the F1 scores of classes present in the evaluation partition.
- **Per-class classification reports and confusion matrices:** Evidence of which labels are recognized reliably and which are confused.
- **Learning curves:** Training and validation loss, accuracy, and macro-F1 used to examine convergence and overfitting.
- **Calibration metrics:** Confidence reliability of the selected model, assessed after selection using separate validation groups.

Gender labels describe the catalogue category rather than the identity of a person in an image. Occasion labels may overlap visually, and frequent categories can dominate accuracy. Each target is trained, compared, and evaluated separately.

The workflow covers metadata inspection, preprocessing, model development, comparative evaluation, selection, and export for prediction. All candidates use the same frozen group-isolated data partitions. The internal test has prior development exposure, which remains a limitation after retraining. Numerical findings and the final model judgment must be completed from the new executed results; no winner is assumed in advance.


## 2. Library Imports & Setup

Use the Fashion Keras kernel. The seed controls initialization and augmentation. Float32 is used throughout; GPU availability is reported before models are created.


In [ ]:
from pathlib import Path
import sys, json
from concurrent.futures import ThreadPoolExecutor
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageEnhance
from IPython.display import display
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, classification_report, log_loss, ConfusionMatrixDisplay
from sklearn.model_selection import GroupShuffleSplit
from scipy.optimize import minimize_scalar
import tensorflow as tf
from tensorflow import keras
from scripts.preprocessing import task_frame, IMAGE_SIZE, NORMALISATION_PATH, SEED, select_tensorflow_device
DEVICE = select_tensorflow_device()
keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()
OUTPUT, RESULTS, FIGURES = (ROOT / 'models', ROOT / 'results', ROOT / 'figures')
for directory in (OUTPUT, RESULTS, FIGURES):
    directory.mkdir(parents=True, exist_ok=True)
MAX_EPOCHS, BATCH_SIZE = (30, 64)

def stem_for(target):
    return 'article_type' if target == 'articleType' else target
get_ipython().run_line_magic('matplotlib', 'inline')

# False for a fresh Run All. Enable only when deliberately resuming matching saved experiments.
RESUME_SAVED_RESULTS = False

## 3. Load Metadata

Load valid labelled images and their frozen split assignments. Inspect paths, target labels and missing values before preprocessing.


In [ ]:
metadata_by_target = {'gender': task_frame('gender'), 'usage': task_frame('usage')}
frame = metadata_by_target['gender']
print('gender', frame.shape)
display(frame.head())
frame = metadata_by_target['usage']
print('usage', frame.shape)
display(frame.head())

## 4. Data Preprocessing

Prepare the same images and label encoding for every candidate. Preserve group isolation and fit normalization on training data only.


### 4.1. Class Distribution & Balancing Strategy

Preserve the original sample distribution. Initial candidates use unweighted cross-entropy; additional CNN experiments compare this with capped square-root inverse-frequency weights computed only from training counts. No examples are duplicated, and validation metrics remain unweighted.


In [ ]:
train_rows = metadata_by_target['gender'].loc[metadata_by_target['gender']['split'].eq('train')]
counts = train_rows['gender'].value_counts()
counts.head(25).sort_values().plot.barh(figsize=(8, 6), title=f"{'gender'}: training support")
plt.tight_layout()
plt.show()
train_rows = metadata_by_target['usage'].loc[metadata_by_target['usage']['split'].eq('train')]
counts = train_rows['usage'].value_counts()
counts.head(25).sort_values().plot.barh(figsize=(8, 6), title=f"{'usage'}: training support")
plt.tight_layout()
plt.show()

### 4.2. Training, Validation & Test Partitions

Reuse the frozen product-group split. Within validation, use separate groups for selection, temperature fitting and policy checking. Do not use the internal test to select candidates.


In [ ]:
def validation_views(frame):
    selection_ids, rest_ids = next(GroupShuffleSplit(n_splits=1, train_size=0.5, random_state=SEED)
                                   .split(frame, groups=frame.group_key))
    selection, rest = frame.iloc[selection_ids], frame.iloc[rest_ids]
    cal_ids, policy_ids = next(GroupShuffleSplit(n_splits=1, train_size=0.5, random_state=SEED + 1)
                              .split(rest, groups=rest.group_key))
    return selection, rest.iloc[cal_ids], rest.iloc[policy_ids]

In [ ]:
frames, labels_by_target = ({}, {})
selection, calibration, policy = validation_views(metadata_by_target['gender'].loc[metadata_by_target['gender']['split'].eq('validation')])
frames['gender'] = dict(train=metadata_by_target['gender'].loc[metadata_by_target['gender']['split'].eq('train')], selection=selection, calibration=calibration, policy=policy)
groups = [set(frame.group_key) for frame in frames['gender'].values()]
assert all((a.isdisjoint(b) for i, a in enumerate(groups) for b in groups[i + 1:]))
labels_by_target['gender'] = sorted(frames['gender']['train']['gender'].unique())
display(pd.Series({name: len(frame) for name, frame in frames['gender'].items()}, name='gender'))
selection, calibration, policy = validation_views(metadata_by_target['usage'].loc[metadata_by_target['usage']['split'].eq('validation')])
frames['usage'] = dict(train=metadata_by_target['usage'].loc[metadata_by_target['usage']['split'].eq('train')], selection=selection, calibration=calibration, policy=policy)
groups = [set(frame.group_key) for frame in frames['usage'].values()]
assert all((a.isdisjoint(b) for i, a in enumerate(groups) for b in groups[i + 1:]))
labels_by_target['usage'] = sorted(frames['usage']['train']['usage'].unique())
display(pd.Series({name: len(frame) for name, frame in frames['usage'].items()}, name='usage'))

### 4.3. Image Preprocessing

Resize RGB to 96 x 128 pixels (width x height), then normalize using training-only channel statistics. The batch class stores decoded uint8 images and converts one batch at a time to float32. Horizontal flips are applied inside each model only during training.


In [ ]:
class CachedBatches(keras.utils.PyDataset):
    """Decode RGB once to uint8 RAM, then normalize each NHWC batch."""
    def __init__(self, frame, target, labels, normalisation, training=False, batch_size=64):
        super().__init__(workers=0, max_queue_size=2)
        self.dataset = frame
        self.training, self.batch_size = training, batch_size
        def read(path):
            with Image.open(path) as image:
                return np.asarray(image.convert("RGB").resize(IMAGE_SIZE, Image.Resampling.BILINEAR)).copy()
        with ThreadPoolExecutor(max_workers=4) as pool:
            self.images = np.stack(list(pool.map(read, frame.image_path)))
        self.targets = np.asarray([labels.index(label) for label in frame[target]], dtype=np.int32)
        self.mean = np.asarray(normalisation["mean"], dtype=np.float32)
        self.std = np.asarray(normalisation["std"], dtype=np.float32)
        self.reset()

    def reset(self):
        self.rng = np.random.default_rng(SEED)
        self.indices = np.arange(len(self.dataset))
        if self.training:
            self.rng.shuffle(self.indices)

    def __len__(self):
        return (len(self.dataset) + self.batch_size - 1) // self.batch_size

    def __getitem__(self, index):
        ids = self.indices[index * self.batch_size:(index + 1) * self.batch_size]
        images = self.images[ids].astype(np.float32) / 255.0
        return (images - self.mean) / self.std, self.targets[ids]

    def on_epoch_end(self):
        if self.training:
            self.rng.shuffle(self.indices)

### 4.4. Feature Batches & Label Encoding

Map sorted label names to integer indices, preserving the same order for every model. Construct shuffled training batches and fixed-order selection batches.


In [ ]:
normalisation = json.loads(NORMALISATION_PATH.read_text())
loaders = {}
labels = labels_by_target['gender']
loaders['gender'] = {name: CachedBatches(frames['gender'][name], 'gender', labels, normalisation, training=name == 'train', batch_size=BATCH_SIZE) for name in ['train', 'selection']}
labels = labels_by_target['usage']
loaders['usage'] = {name: CachedBatches(frames['usage'][name], 'usage', labels, normalisation, training=name == 'train', batch_size=BATCH_SIZE) for name in ['train', 'selection']}

## 5. Model Development & Evaluation

Train and evaluate a shallow MLP baseline, a deeper MLP and a CNN using the same image preprocessing and frozen partitions. Architecture construction, fitting, class-level evaluation, calibration and model export are implemented in this notebook.


### Evaluation Metric: Macro-F1

Accumulate the full-epoch confusion matrix and average F1 over classes with positive ground-truth support. This metric selects the restored epoch and winning model. It does not average per-batch F1 scores.

The displayed per-class classification report also includes every output label for coverage auditing. Its standard `macro avg` row therefore includes zero-support output labels and can differ from the supported-class macro-F1 used for selection. Read the explicitly reported selection metric for model ranking, and inspect support before making claims about all classes. Selection support is 97/124 article types, 4/4 seasons, 5/5 gender categories and 8/9 occasion labels for this frozen run.


In [ ]:
@keras.utils.register_keras_serializable(package="Fashion")
class SupportedMacroF1(keras.metrics.Metric):
    """Macro-F1 over classes present in the full epoch's ground truth."""
    def __init__(self, num_classes, name="macro_f1", **kwargs):
        super().__init__(name=name, **kwargs)
        self.num_classes = num_classes
        self.matrix = self.add_weight(name="matrix", shape=(num_classes, num_classes), initializer="zeros")

    def update_state(self, y_true, y_pred, sample_weight=None):
        truth = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
        predictions = tf.argmax(y_pred, axis=-1, output_type=tf.int32)
        weights = None if sample_weight is None else tf.cast(tf.reshape(sample_weight, [-1]), self.dtype)
        self.matrix.assign_add(tf.math.confusion_matrix(truth, predictions, self.num_classes,
                                                        weights=weights, dtype=self.dtype))

    def result(self):
        support = tf.reduce_sum(self.matrix, axis=1)
        predicted = tf.reduce_sum(self.matrix, axis=0)
        f1 = tf.math.divide_no_nan(2 * tf.linalg.diag_part(self.matrix), support + predicted)
        mask = tf.cast(support > 0, self.dtype)
        return tf.math.divide_no_nan(tf.reduce_sum(f1 * mask), tf.reduce_sum(mask))

    def reset_state(self):
        self.matrix.assign(tf.zeros_like(self.matrix))

    def get_config(self):
        return {**super().get_config(), "num_classes": self.num_classes}

### Saved-Model Metadata

An identity layer stores label order, preprocessing and confidence policy inside each exported Keras model.


In [ ]:
@keras.utils.register_keras_serializable(package="Fashion")
class ModelMetadata(keras.layers.Layer):
    """Store label order, preprocessing and calibration inside the .keras file."""
    def __init__(self, metadata=None, **kwargs):
        super().__init__(**kwargs)
        self.metadata = dict(metadata or {})

    def call(self, inputs):
        return inputs

    def get_config(self):
        return {**super().get_config(), "metadata": dict(self.metadata)}

In [ ]:
models = {'gender': {}, 'usage': {}}
histories = {'gender': {}, 'usage': {}}

### Configuration rationale and evaluation protocol

Each target compares three neural model families. Dense width and depth control model capacity; convolutional blocks preserve spatial relationships. Configuration choices follow validation macro-F1, then accuracy, then parameter count. The selected architectures and training settings are specified below; they are empirical choices, not theoretically optimal designs.

The 96 ? 128 RGB input and training-only normalization are shared across candidates. Most source images are only 60 ? 80, so enlarging them further would not recover additional image detail. Output width is determined by the target's training label vocabulary, rather than tuned independently.

Where required, CNN training has an initial four-block stage followed by a lower-learning-rate continuation. Both stages are implemented here. Class weights use training counts only. The initial stage is part of the training recipe, not a fourth model family.

The recorded tables summarize completed runs of these configurations. This reorganized notebook has not yet been executed end-to-end; rerunning the cells regenerates its metrics, figures and exports. Single-seed results may vary. Only the supplied dataset, shared preprocessing module and frozen Task 0 preprocessing artifacts are required; all model training code is included here.


### General design justification

**Input and output.** A fixed 96 ? 128 RGB input standardizes batching and inference while keeping computation manageable. Its portrait aspect ratio matches the dominant 60 ? 80 source format. Resizing cannot recover missing detail, so larger inputs are not assumed to improve recognition. Output units match the target's label vocabulary; they are not a freely tuned capacity setting.

**Dense baselines.** Flattening provides a straightforward pixel-based baseline. A shallow MLP tests whether one nonlinear hidden layer is sufficient; a deeper MLP tests additional nonlinear transformations. Dense layers do not explicitly preserve local spatial relationships, and flattening creates large parameter counts. ReLU introduces nonlinearity. Hidden widths and depths are selected using validation evidence, not the number of classes alone.

**CNN architecture.** Small 3 ? 3 convolutions share weights across positions and can learn local appearance patterns. Successive blocks allow information from larger image regions to be combined. Max pooling reduces spatial dimensions and computational cost, but can discard fine detail. The final 2 ? 2 average-pooled grid limits the dense head's size while retaining a coarse spatial layout. Batch normalization stabilizes intermediate activation scales. These design properties motivate CNN use; they do not prove that a particular learned filter detects a specific garment feature.

**Regularization and optimization.** Dropout 0.2 discourages reliance on individual hidden activations, while training-only horizontal flips expose models to mirrored product views. These are attempts to improve generalization, not guarantees. Adam adapts parameter updates; reducing its learning rate on a validation plateau allows smaller updates later in CNN training. Early stopping restores the best selection macro-F1 epoch and limits training after improvement stalls. The fixed seed supports repeatability, and batch size 64 is a practical training setting rather than a demonstrated optimum.

**Model selection.** Group-isolated partitions reduce related-product leakage, training-only normalization avoids fitting preprocessing to evaluation data, and macro-F1 gives each supported class equal weight. Accuracy, class-level errors and parameter counts provide complementary evidence. Different labels may favour different capacities; the measured validation comparison justifies the exact choices, while the general principles above explain their intended roles.


In [ ]:
SELECTED_CONFIGS = {'gender': {'shallow_mlp': {'method': 'shallow_mlp_lower_lr', 'widths': [256], 'learning_rate': 0.0001, 'epochs': 20, 'patience': 4}, 'deeper_mlp': {'method': 'deeper_mlp_two_layers', 'widths': [256, 128], 'learning_rate': 0.0003, 'epochs': 20, 'patience': 4}, 'cnn': {'method': 'cnn_four_blocks_scheduled', 'widths': [256], 'learning_rate': 0.001, 'epochs': 30, 'patience': 7}}, 'usage': {'shallow_mlp': {'method': 'shallow_mlp_lower_lr', 'widths': [256], 'learning_rate': 0.0001, 'epochs': 20, 'patience': 4}, 'deeper_mlp': {'method': 'deeper_mlp', 'widths': [256, 128, 64], 'learning_rate': 0.001, 'epochs': 30, 'patience': 7}, 'cnn': {'method': 'cnn_continue_weighted', 'widths': [256], 'learning_rate': 3e-05, 'epochs': 30, 'patience': 7}}}

### 5.1. Shallow MLP (Baseline)

Flattened RGB input is passed through the selected dense layers, with ReLU and dropout 0.2 after each hidden layer.


#### gender: architecture and training settings

Configuration `shallow_mlp_lower_lr`; hidden dense units [256]. Adam 0.0001, maximum 20 epochs, early-stopping patience 4.


In [ ]:
config = SELECTED_CONFIGS['gender']['shallow_mlp']
method = config['method']
keras.utils.set_random_seed(SEED)
layers = [keras.Input(shape=(IMAGE_SIZE[1], IMAGE_SIZE[0], 3)), keras.layers.RandomFlip('horizontal')]
layers.append(keras.layers.Flatten())
for width in config['widths']:
    layers.extend([keras.layers.Dense(width, activation='relu'), keras.layers.Dropout(0.2)])
layers.extend([keras.layers.Dense(len(labels_by_target['gender'])), ModelMetadata(name='metadata')])
models['gender'][method] = keras.Sequential(layers, name=method)
models['gender'][method].summary()

#### Compile and train

The selection view controls epoch selection. Test images are excluded from training.


In [ ]:
model = models['gender'][method]
model.compile(optimizer=keras.optimizers.Adam(0.0001), loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=[keras.metrics.SparseCategoricalAccuracy(name='accuracy'), SupportedMacroF1(len(labels_by_target['gender']))])
loaders['gender']['train'].reset()
callbacks = [keras.callbacks.EarlyStopping(monitor='val_macro_f1', mode='max', patience=4, restore_best_weights=True)]
fitted = model.fit(loaders['gender']['train'], validation_data=loaders['gender']['selection'], epochs=20, callbacks=callbacks, shuffle=False, verbose=2)
histories['gender'][method] = pd.DataFrame(fitted.history)

In [ ]:
loader = loaders['gender']['selection']
probabilities = np.vstack([tf.nn.softmax(models['gender'][method](loader[i][0], training=False)).numpy() for i in range(len(loader))])
predictions = probabilities.argmax(1)
display(pd.Series(dict(selection_accuracy=accuracy_score(loader.targets, predictions), selection_macro_f1=f1_score(loader.targets, predictions, labels=np.unique(loader.targets), average='macro', zero_division=0))))
per_class = pd.DataFrame(classification_report(loader.targets, predictions, labels=np.arange(len(labels_by_target['gender'])), target_names=labels_by_target['gender'], output_dict=True, zero_division=0)).T
display(per_class)
per_class.to_csv(RESULTS / f"{stem_for('gender')}_{method}_selection_per_class.csv")
history = histories['gender'][method]
history.to_csv(RESULTS / f"{stem_for('gender')}_{method}_history.csv", index=False)
fig, axes = plt.subplots(1, 3, figsize=(14, 3))
for ax, metric in zip(axes, ['loss', 'accuracy', 'macro_f1']):
    ax.plot(np.arange(1, len(history)+1), history[metric], label='Train')
    ax.plot(np.arange(1, len(history)+1), history['val_'+metric], label='Selection')
    ax.set(xlabel='Epoch', ylabel=metric, title=method)
    ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / f"{stem_for('gender')}_{method}_learning_curves.png", dpi=160)
plt.show()

#### Recorded evaluation: gender

The completed experiment achieved **84.69% selection accuracy** and **0.7118 macro-F1**, using **9,438,725 parameters**. At its best macro-F1 epoch, training accuracy was 85.00% and validation loss was 0.4602. Training uses augmentation and dropout; weighted training loss, where applicable, is not directly comparable with unweighted validation loss.

The preceding evaluation cell displays per-class precision, recall and F1 and saves its table in `results/` when run. Aggregate accuracy does not establish rare-class reliability.


#### usage: architecture and training settings

Configuration `shallow_mlp_lower_lr`; hidden dense units [256]. Adam 0.0001, maximum 20 epochs, early-stopping patience 4.


In [ ]:
config = SELECTED_CONFIGS['usage']['shallow_mlp']
method = config['method']
keras.utils.set_random_seed(SEED)
layers = [keras.Input(shape=(IMAGE_SIZE[1], IMAGE_SIZE[0], 3)), keras.layers.RandomFlip('horizontal')]
layers.append(keras.layers.Flatten())
for width in config['widths']:
    layers.extend([keras.layers.Dense(width, activation='relu'), keras.layers.Dropout(0.2)])
layers.extend([keras.layers.Dense(len(labels_by_target['usage'])), ModelMetadata(name='metadata')])
models['usage'][method] = keras.Sequential(layers, name=method)
models['usage'][method].summary()

#### Compile and train

The selection view controls epoch selection. Test images are excluded from training.


In [ ]:
model = models['usage'][method]
model.compile(optimizer=keras.optimizers.Adam(0.0001), loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=[keras.metrics.SparseCategoricalAccuracy(name='accuracy'), SupportedMacroF1(len(labels_by_target['usage']))])
loaders['usage']['train'].reset()
callbacks = [keras.callbacks.EarlyStopping(monitor='val_macro_f1', mode='max', patience=4, restore_best_weights=True)]
fitted = model.fit(loaders['usage']['train'], validation_data=loaders['usage']['selection'], epochs=20, callbacks=callbacks, shuffle=False, verbose=2)
histories['usage'][method] = pd.DataFrame(fitted.history)

In [ ]:
loader = loaders['usage']['selection']
probabilities = np.vstack([tf.nn.softmax(models['usage'][method](loader[i][0], training=False)).numpy() for i in range(len(loader))])
predictions = probabilities.argmax(1)
display(pd.Series(dict(selection_accuracy=accuracy_score(loader.targets, predictions), selection_macro_f1=f1_score(loader.targets, predictions, labels=np.unique(loader.targets), average='macro', zero_division=0))))
per_class = pd.DataFrame(classification_report(loader.targets, predictions, labels=np.arange(len(labels_by_target['usage'])), target_names=labels_by_target['usage'], output_dict=True, zero_division=0)).T
display(per_class)
per_class.to_csv(RESULTS / f"{stem_for('usage')}_{method}_selection_per_class.csv")
history = histories['usage'][method]
history.to_csv(RESULTS / f"{stem_for('usage')}_{method}_history.csv", index=False)
fig, axes = plt.subplots(1, 3, figsize=(14, 3))
for ax, metric in zip(axes, ['loss', 'accuracy', 'macro_f1']):
    ax.plot(np.arange(1, len(history)+1), history[metric], label='Train')
    ax.plot(np.arange(1, len(history)+1), history['val_'+metric], label='Selection')
    ax.set(xlabel='Epoch', ylabel=metric, title=method)
    ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / f"{stem_for('usage')}_{method}_learning_curves.png", dpi=160)
plt.show()

#### Recorded evaluation: usage

The completed experiment achieved **87.38% selection accuracy** and **0.4522 macro-F1**, using **9,439,753 parameters**. At its best macro-F1 epoch, training accuracy was 86.81% and validation loss was 0.4874. Training uses augmentation and dropout; weighted training loss, where applicable, is not directly comparable with unweighted validation loss.

The preceding evaluation cell displays per-class precision, recall and F1 and saves its table in `results/` when run. Aggregate accuracy does not establish rare-class reliability.


### 5.2. Deeper MLP

Flattened RGB input is passed through the selected dense layers, with ReLU and dropout 0.2 after each hidden layer.


#### gender: architecture and training settings

Configuration `deeper_mlp_two_layers`; hidden dense units [256, 128]. Adam 0.0003, maximum 20 epochs, early-stopping patience 4.


In [ ]:
config = SELECTED_CONFIGS['gender']['deeper_mlp']
method = config['method']
keras.utils.set_random_seed(SEED)
layers = [keras.Input(shape=(IMAGE_SIZE[1], IMAGE_SIZE[0], 3)), keras.layers.RandomFlip('horizontal')]
layers.append(keras.layers.Flatten())
for width in config['widths']:
    layers.extend([keras.layers.Dense(width, activation='relu'), keras.layers.Dropout(0.2)])
layers.extend([keras.layers.Dense(len(labels_by_target['gender'])), ModelMetadata(name='metadata')])
models['gender'][method] = keras.Sequential(layers, name=method)
models['gender'][method].summary()

#### Compile and train

The selection view controls epoch selection. Test images are excluded from training.


In [ ]:
model = models['gender'][method]
model.compile(optimizer=keras.optimizers.Adam(0.0003), loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=[keras.metrics.SparseCategoricalAccuracy(name='accuracy'), SupportedMacroF1(len(labels_by_target['gender']))])
loaders['gender']['train'].reset()
callbacks = [keras.callbacks.EarlyStopping(monitor='val_macro_f1', mode='max', patience=4, restore_best_weights=True)]
fitted = model.fit(loaders['gender']['train'], validation_data=loaders['gender']['selection'], epochs=20, callbacks=callbacks, shuffle=False, verbose=2)
histories['gender'][method] = pd.DataFrame(fitted.history)

In [ ]:
loader = loaders['gender']['selection']
probabilities = np.vstack([tf.nn.softmax(models['gender'][method](loader[i][0], training=False)).numpy() for i in range(len(loader))])
predictions = probabilities.argmax(1)
display(pd.Series(dict(selection_accuracy=accuracy_score(loader.targets, predictions), selection_macro_f1=f1_score(loader.targets, predictions, labels=np.unique(loader.targets), average='macro', zero_division=0))))
per_class = pd.DataFrame(classification_report(loader.targets, predictions, labels=np.arange(len(labels_by_target['gender'])), target_names=labels_by_target['gender'], output_dict=True, zero_division=0)).T
display(per_class)
per_class.to_csv(RESULTS / f"{stem_for('gender')}_{method}_selection_per_class.csv")
history = histories['gender'][method]
history.to_csv(RESULTS / f"{stem_for('gender')}_{method}_history.csv", index=False)
fig, axes = plt.subplots(1, 3, figsize=(14, 3))
for ax, metric in zip(axes, ['loss', 'accuracy', 'macro_f1']):
    ax.plot(np.arange(1, len(history)+1), history[metric], label='Train')
    ax.plot(np.arange(1, len(history)+1), history['val_'+metric], label='Selection')
    ax.set(xlabel='Epoch', ylabel=metric, title=method)
    ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / f"{stem_for('gender')}_{method}_learning_curves.png", dpi=160)
plt.show()

#### Recorded evaluation: gender

The completed experiment achieved **84.52% selection accuracy** and **0.6912 macro-F1**, using **9,470,981 parameters**. At its best macro-F1 epoch, training accuracy was 84.07% and validation loss was 0.4664. Training uses augmentation and dropout; weighted training loss, where applicable, is not directly comparable with unweighted validation loss.

The preceding evaluation cell displays per-class precision, recall and F1 and saves its table in `results/` when run. Aggregate accuracy does not establish rare-class reliability.


#### usage: architecture and training settings

Configuration `deeper_mlp`; hidden dense units [256, 128, 64]. Adam 0.001, maximum 30 epochs, early-stopping patience 7.


In [ ]:
config = SELECTED_CONFIGS['usage']['deeper_mlp']
method = config['method']
keras.utils.set_random_seed(SEED)
layers = [keras.Input(shape=(IMAGE_SIZE[1], IMAGE_SIZE[0], 3)), keras.layers.RandomFlip('horizontal')]
layers.append(keras.layers.Flatten())
for width in config['widths']:
    layers.extend([keras.layers.Dense(width, activation='relu'), keras.layers.Dropout(0.2)])
layers.extend([keras.layers.Dense(len(labels_by_target['usage'])), ModelMetadata(name='metadata')])
models['usage'][method] = keras.Sequential(layers, name=method)
models['usage'][method].summary()

#### Compile and train

The selection view controls epoch selection. Test images are excluded from training.


In [ ]:
model = models['usage'][method]
model.compile(optimizer=keras.optimizers.Adam(0.001), loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=[keras.metrics.SparseCategoricalAccuracy(name='accuracy'), SupportedMacroF1(len(labels_by_target['usage']))])
loaders['usage']['train'].reset()
callbacks = [keras.callbacks.EarlyStopping(monitor='val_macro_f1', mode='max', patience=7, restore_best_weights=True)]
fitted = model.fit(loaders['usage']['train'], validation_data=loaders['usage']['selection'], epochs=30, callbacks=callbacks, shuffle=False, verbose=2)
histories['usage'][method] = pd.DataFrame(fitted.history)

In [ ]:
loader = loaders['usage']['selection']
probabilities = np.vstack([tf.nn.softmax(models['usage'][method](loader[i][0], training=False)).numpy() for i in range(len(loader))])
predictions = probabilities.argmax(1)
display(pd.Series(dict(selection_accuracy=accuracy_score(loader.targets, predictions), selection_macro_f1=f1_score(loader.targets, predictions, labels=np.unique(loader.targets), average='macro', zero_division=0))))
per_class = pd.DataFrame(classification_report(loader.targets, predictions, labels=np.arange(len(labels_by_target['usage'])), target_names=labels_by_target['usage'], output_dict=True, zero_division=0)).T
display(per_class)
per_class.to_csv(RESULTS / f"{stem_for('usage')}_{method}_selection_per_class.csv")
history = histories['usage'][method]
history.to_csv(RESULTS / f"{stem_for('usage')}_{method}_history.csv", index=False)
fig, axes = plt.subplots(1, 3, figsize=(14, 3))
for ax, metric in zip(axes, ['loss', 'accuracy', 'macro_f1']):
    ax.plot(np.arange(1, len(history)+1), history[metric], label='Train')
    ax.plot(np.arange(1, len(history)+1), history['val_'+metric], label='Selection')
    ax.set(xlabel='Epoch', ylabel=metric, title=method)
    ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / f"{stem_for('usage')}_{method}_learning_curves.png", dpi=160)
plt.show()

#### Recorded evaluation: usage

The completed experiment achieved **86.55% selection accuracy** and **0.3694 macro-F1**, using **9,479,177 parameters**. At its best macro-F1 epoch, training accuracy was 86.42% and validation loss was 0.4069. Training uses augmentation and dropout; weighted training loss, where applicable, is not directly comparable with unweighted validation loss.

The preceding evaluation cell displays per-class precision, recall and F1 and saves its table in `results/` when run. Aggregate accuracy does not establish rare-class reliability.


### 5.3. Convolutional Neural Network (CNN)

The selected CNN uses four 3 ? 3 convolution blocks (32, 64, 128, 256 filters), batch normalization, ReLU and max pooling, followed by a 2 ? 2 pooled grid and Dense(256). Only the selected continuation, when required, is fitted.


#### gender: architecture and training settings

Configuration `cnn_four_blocks_scheduled`; hidden dense units [256]. Initial Adam 0.001, maximum 30 epochs, early-stopping patience 7 and learning-rate halving after two unimproved epochs. Continuation, if specified, uses eight additional epochs with patience 4 and a fresh optimizer.


In [ ]:
config = SELECTED_CONFIGS['gender']['cnn']
method = config['method']
keras.utils.set_random_seed(SEED)
layers = [keras.Input(shape=(IMAGE_SIZE[1], IMAGE_SIZE[0], 3)), keras.layers.RandomFlip('horizontal')]
for width in [32, 64, 128, 256]:
    layers.extend([keras.layers.Conv2D(width, 3, padding='same'), keras.layers.BatchNormalization(), keras.layers.Activation('relu'), keras.layers.MaxPooling2D(2)])
h, w = IMAGE_SIZE[1] // 16, IMAGE_SIZE[0] // 16
layers.extend([keras.layers.AveragePooling2D((h // 2, w // 2)), keras.layers.Flatten()])
for width in config['widths']:
    layers.extend([keras.layers.Dense(width, activation='relu'), keras.layers.Dropout(0.2)])
layers.extend([keras.layers.Dense(len(labels_by_target['gender'])), ModelMetadata(name='metadata')])
models['gender'][method] = keras.Sequential(layers, name=method)
models['gender'][method].summary()

#### Compile and train

The selection view controls epoch selection. Test images are excluded from training.


In [ ]:
model = models['gender'][method]
model.compile(optimizer=keras.optimizers.Adam(0.001), loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=[keras.metrics.SparseCategoricalAccuracy(name='accuracy'), SupportedMacroF1(len(labels_by_target['gender']))])
loaders['gender']['train'].reset()
callbacks = [keras.callbacks.EarlyStopping(monitor='val_macro_f1', mode='max', patience=7, restore_best_weights=True)]
callbacks.insert(0, keras.callbacks.ReduceLROnPlateau(monitor='val_macro_f1', mode='max', factor=0.5, patience=2))
fitted = model.fit(loaders['gender']['train'], validation_data=loaders['gender']['selection'], epochs=30, callbacks=callbacks, shuffle=False, verbose=2)
histories['gender'][method] = pd.DataFrame(fitted.history)

In [ ]:
loader = loaders['gender']['selection']
probabilities = np.vstack([tf.nn.softmax(models['gender'][method](loader[i][0], training=False)).numpy() for i in range(len(loader))])
predictions = probabilities.argmax(1)
display(pd.Series(dict(selection_accuracy=accuracy_score(loader.targets, predictions), selection_macro_f1=f1_score(loader.targets, predictions, labels=np.unique(loader.targets), average='macro', zero_division=0))))
per_class = pd.DataFrame(classification_report(loader.targets, predictions, labels=np.arange(len(labels_by_target['gender'])), target_names=labels_by_target['gender'], output_dict=True, zero_division=0)).T
display(per_class)
per_class.to_csv(RESULTS / f"{stem_for('gender')}_{method}_selection_per_class.csv")
history = histories['gender'][method]
history.to_csv(RESULTS / f"{stem_for('gender')}_{method}_history.csv", index=False)
fig, axes = plt.subplots(1, 3, figsize=(14, 3))
for ax, metric in zip(axes, ['loss', 'accuracy', 'macro_f1']):
    ax.plot(np.arange(1, len(history)+1), history[metric], label='Train')
    ax.plot(np.arange(1, len(history)+1), history['val_'+metric], label='Selection')
    ax.set(xlabel='Epoch', ylabel=metric, title=method)
    ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / f"{stem_for('gender')}_{method}_learning_curves.png", dpi=160)
plt.show()

#### Recorded evaluation: gender

The completed experiment achieved **89.86% selection accuracy** and **0.7909 macro-F1**, using **654,021 parameters**. At its best macro-F1 epoch, training accuracy was 96.81% and validation loss was 0.3662. Training uses augmentation and dropout; weighted training loss, where applicable, is not directly comparable with unweighted validation loss.

The preceding evaluation cell displays per-class precision, recall and F1 and saves its table in `results/` when run. Aggregate accuracy does not establish rare-class reliability.


#### usage: architecture and training settings

Configuration `cnn_continue_weighted`; hidden dense units [256]. Initial Adam 0.001, maximum 30 epochs, early-stopping patience 7 and learning-rate halving after two unimproved epochs. Continuation, if specified, uses eight additional epochs with patience 4 and a fresh optimizer.


In [ ]:
config = SELECTED_CONFIGS['usage']['cnn']
method = config['method']
keras.utils.set_random_seed(SEED)
layers = [keras.Input(shape=(IMAGE_SIZE[1], IMAGE_SIZE[0], 3)), keras.layers.RandomFlip('horizontal')]
for width in [32, 64, 128, 256]:
    layers.extend([keras.layers.Conv2D(width, 3, padding='same'), keras.layers.BatchNormalization(), keras.layers.Activation('relu'), keras.layers.MaxPooling2D(2)])
h, w = IMAGE_SIZE[1] // 16, IMAGE_SIZE[0] // 16
layers.extend([keras.layers.AveragePooling2D((h // 2, w // 2)), keras.layers.Flatten()])
for width in config['widths']:
    layers.extend([keras.layers.Dense(width, activation='relu'), keras.layers.Dropout(0.2)])
layers.extend([keras.layers.Dense(len(labels_by_target['usage'])), ModelMetadata(name='metadata')])
models['usage'][method] = keras.Sequential(layers, name=method)
models['usage'][method].summary()

#### Compile and train

The selection view controls epoch selection. Test images are excluded from training.


In [ ]:
model = models['usage'][method]
model.compile(optimizer=keras.optimizers.Adam(0.001), loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=[keras.metrics.SparseCategoricalAccuracy(name='accuracy'), SupportedMacroF1(len(labels_by_target['usage']))])
loaders['usage']['train'].reset()
callbacks = [keras.callbacks.EarlyStopping(monitor='val_macro_f1', mode='max', patience=7, restore_best_weights=True)]
callbacks.insert(0, keras.callbacks.ReduceLROnPlateau(monitor='val_macro_f1', mode='max', factor=0.5, patience=2))
fitted = model.fit(loaders['usage']['train'], validation_data=loaders['usage']['selection'], epochs=30, callbacks=callbacks, shuffle=False, verbose=2)
histories['usage'][method] = pd.DataFrame(fitted.history)
histories['usage'][method].to_csv(RESULTS / 'usage_cnn_initial_history.csv', index=False)
keras.utils.set_random_seed(SEED)
loaders['usage']['train'].reset()
model.compile(optimizer=keras.optimizers.Adam(3e-05), loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=[keras.metrics.SparseCategoricalAccuracy(name='accuracy'), SupportedMacroF1(len(labels_by_target['usage']))])
class_weight = None
counts = np.bincount(loaders['usage']['train'].targets, minlength=len(labels_by_target['usage']))
weights = np.sqrt(counts.sum() / (len(counts) * counts))
weights = np.clip(weights / np.average(weights, weights=counts), 0.25, 6)
class_weight = dict(enumerate(weights))
fitted = model.fit(loaders['usage']['train'], validation_data=loaders['usage']['selection'], epochs=8, callbacks=[keras.callbacks.EarlyStopping(monitor='val_macro_f1', mode='max', patience=4, restore_best_weights=True)], class_weight=class_weight, shuffle=False, verbose=2)
histories['usage'][method] = pd.DataFrame(fitted.history)

In [ ]:
loader = loaders['usage']['selection']
probabilities = np.vstack([tf.nn.softmax(models['usage'][method](loader[i][0], training=False)).numpy() for i in range(len(loader))])
predictions = probabilities.argmax(1)
display(pd.Series(dict(selection_accuracy=accuracy_score(loader.targets, predictions), selection_macro_f1=f1_score(loader.targets, predictions, labels=np.unique(loader.targets), average='macro', zero_division=0))))
per_class = pd.DataFrame(classification_report(loader.targets, predictions, labels=np.arange(len(labels_by_target['usage'])), target_names=labels_by_target['usage'], output_dict=True, zero_division=0)).T
display(per_class)
per_class.to_csv(RESULTS / f"{stem_for('usage')}_{method}_selection_per_class.csv")
history = histories['usage'][method]
history.to_csv(RESULTS / f"{stem_for('usage')}_{method}_history.csv", index=False)
fig, axes = plt.subplots(1, 3, figsize=(14, 3))
for ax, metric in zip(axes, ['loss', 'accuracy', 'macro_f1']):
    ax.plot(np.arange(1, len(history)+1), history[metric], label='Train')
    ax.plot(np.arange(1, len(history)+1), history['val_'+metric], label='Selection')
    ax.set(xlabel='Epoch', ylabel=metric, title=method)
    ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / f"{stem_for('usage')}_{method}_learning_curves.png", dpi=160)
plt.show()

#### Recorded evaluation: usage

The completed experiment achieved **89.55% selection accuracy** and **0.4497 macro-F1**, using **655,049 parameters**. At its best macro-F1 epoch, training accuracy was 95.26% and validation loss was 0.3138. Training uses augmentation and dropout; weighted training loss, where applicable, is not directly comparable with unweighted validation loss.

The preceding evaluation cell displays per-class precision, recall and F1 and saves its table in `results/` when run. Aggregate accuracy does not establish rare-class reliability.


### 5.4. Model Comparison

Evaluate all three configurations on the same selection view. Rank them by supported-class macro-F1, then accuracy, then parameter count. The table records the chosen architecture and measured performance for each family.


In [ ]:
def supported_macro_f1(truth, predictions) -> float:
    """Macro-average over labels present in the ground-truth partition."""
    return float(f1_score(
        truth, predictions, labels=np.unique(truth), average="macro", zero_division=0
    ))

def expected_calibration_error(
    truth: np.ndarray,
    probabilities: np.ndarray,
    bins: int = 10,
) -> float:
    """Compute top-label expected calibration error."""
    truth = np.asarray(truth)
    probabilities = np.asarray(probabilities)
    confidence = probabilities.max(axis=1)
    correct = probabilities.argmax(axis=1) == truth
    edges = np.linspace(0.0, 1.0, bins + 1)
    total = len(truth)
    error = 0.0
    for lower, upper in zip(edges[:-1], edges[1:], strict=True):
        selected = (confidence > lower) & (confidence <= upper)
        if selected.any():
            error += selected.sum() / total * abs(
                float(correct[selected].mean()) - float(confidence[selected].mean())
            )
    return float(error)

def ranked(table):
    """Predeclared: macro F1, then accuracy, then fewer parameters."""
    return table.sort_values(['validation_macro_f1', 'validation_accuracy', 'complexity_parameters'],
                             ascending=[False, False, True], kind='stable')

In [ ]:
comparisons, tuning, winners, checkpoints = ({}, {}, {}, {})

In [ ]:
rows = []
loader = loaders['gender']['selection']
for family, config in SELECTED_CONFIGS['gender'].items():
    method = config['method']
    model = models['gender'][method]
    probabilities = np.vstack([tf.nn.softmax(model(loader[i][0], training=False)).numpy() for i in range(len(loader))])
    rows.append(dict(method=method, family=family, validation_accuracy=accuracy_score(loader.targets, probabilities.argmax(1)), validation_macro_f1=supported_macro_f1(loader.targets, probabilities.argmax(1)), validation_ece=expected_calibration_error(loader.targets, probabilities), complexity_parameters=model.count_params(), epochs_run=len(histories['gender'][method])))
table = pd.DataFrame(rows).set_index('method')
winners['gender'] = ranked(table).index[0]
table['selected'] = table.index == winners['gender']
comparisons['gender'] = table
tuning['gender'] = table.loc[table.family.eq('cnn')].copy()
table.to_csv(RESULTS / 'gender_comparison.csv')
display(table)

**Recorded model comparison: gender**

| Family | Configuration | Validation accuracy | Macro-F1 | Parameters |
|---|---|---:|---:|---:|
| shallow_mlp | shallow_mlp_lower_lr | 84.69% | 0.7118 | 9,438,725 |
| deeper_mlp | deeper_mlp_two_layers | 84.52% | 0.6912 | 9,470,981 |
| cnn | cnn_four_blocks_scheduled | 89.86% | 0.7909 | 654,021 |


In [ ]:
rows = []
loader = loaders['usage']['selection']
for family, config in SELECTED_CONFIGS['usage'].items():
    method = config['method']
    model = models['usage'][method]
    probabilities = np.vstack([tf.nn.softmax(model(loader[i][0], training=False)).numpy() for i in range(len(loader))])
    rows.append(dict(method=method, family=family, validation_accuracy=accuracy_score(loader.targets, probabilities.argmax(1)), validation_macro_f1=supported_macro_f1(loader.targets, probabilities.argmax(1)), validation_ece=expected_calibration_error(loader.targets, probabilities), complexity_parameters=model.count_params(), epochs_run=len(histories['usage'][method])))
table = pd.DataFrame(rows).set_index('method')
winners['usage'] = ranked(table).index[0]
table['selected'] = table.index == winners['usage']
comparisons['usage'] = table
tuning['usage'] = table.loc[table.family.eq('cnn')].copy()
table.to_csv(RESULTS / 'usage_comparison.csv')
display(table)

**Recorded model comparison: usage**

| Family | Configuration | Validation accuracy | Macro-F1 | Parameters |
|---|---|---:|---:|---:|
| shallow_mlp | shallow_mlp_lower_lr | 87.38% | 0.4522 | 9,439,753 |
| deeper_mlp | deeper_mlp | 86.55% | 0.3694 | 9,479,177 |
| cnn | cnn_continue_weighted | 89.55% | 0.4497 | 655,049 |


### 5.5. Learning Curves

Inspect the restored best epoch and later trajectory rather than the final epoch alone. Training enables augmentation and dropout; selection uses inference mode, so a negative training-minus-selection accuracy gap is possible.

The class-weighted CNN uses weighted training loss but unweighted validation loss. Their raw loss gap does not directly measure overfitting. Accuracy and supported-class macro-F1 remain unweighted and are comparable across candidates, subject to the training/inference-mode difference. CNN continuation plots count additional epochs from the shared starting checkpoint.


In [ ]:
# Gender
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
for method, history in histories['gender'].items():
    history = history.copy()
    history.insert(0, 'epoch', np.arange(1, len(history) + 1))
    history.to_csv(RESULTS / f"{stem_for('gender')}_{method}_history.csv", index=False)
    for ax, metric in zip(axes, ['loss', 'accuracy', 'macro_f1']):
        ax.plot(history.epoch, history[metric], linestyle='--', alpha=0.6, label=f'{method}: train')
        ax.plot(history.epoch, history[f'val_{metric}'], label=f'{method}: selection')
        ax.set(xlabel='Epoch', ylabel=metric, title='gender')
axes[-1].legend(fontsize=6)
fig.tight_layout()
fig.savefig(FIGURES / f"{stem_for('gender')}_learning_curves.png", dpi=160)

In [ ]:
# Occasion (usage)
plt.show()
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
for method, history in histories['usage'].items():
    history = history.copy()
    history.insert(0, 'epoch', np.arange(1, len(history) + 1))
    history.to_csv(RESULTS / f"{stem_for('usage')}_{method}_history.csv", index=False)
    for ax, metric in zip(axes, ['loss', 'accuracy', 'macro_f1']):
        ax.plot(history.epoch, history[metric], linestyle='--', alpha=0.6, label=f'{method}: train')
        ax.plot(history.epoch, history[f'val_{metric}'], label=f'{method}: selection')
        ax.set(xlabel='Epoch', ylabel=metric, title='usage')
axes[-1].legend(fontsize=6)
fig.tight_layout()
fig.savefig(FIGURES / f"{stem_for('usage')}_learning_curves.png", dpi=160)
plt.show()

### 5.6. Model Evaluation & Error Analysis

The following small inference helpers reproduce application preprocessing. Temperature scaling changes confidence, not the predicted class. Calibration is fitted on separate groups, then accepted only if policy-group NLL improves without worsening ECE.


#### 5.6.1. Prepare one inference image

Use the same RGB resizing and training-only normalization as the training batches.


In [ ]:
def image_batch(image, image_size, mean, std):
    resized = image.convert("RGB").resize(tuple(image_size), Image.Resampling.BILINEAR)
    array = np.asarray(resized, dtype=np.float32) / 255.0
    array = (array - np.asarray(mean, dtype=np.float32)) / np.asarray(std, dtype=np.float32)
    return array[None, ...]

#### 5.6.2. Rescale confidence

Divide log probabilities by a positive temperature and renormalize.


In [ ]:
def temperature_scale(probabilities, temperature=1.0):
    """Rescale confidence without changing the highest-probability class."""
    if not np.isfinite(temperature) or temperature <= 0:
        raise ValueError('Temperature must be finite and positive')
    values = np.asarray(probabilities, dtype=np.float64)
    if values.ndim < 1 or values.shape[-1] == 0 or not np.isfinite(values).all() or (values < 0).any():
        raise ValueError('Probabilities must be finite, nonnegative vectors')
    mass = values.sum(axis=-1, keepdims=True)
    if (mass <= 0).any():
        raise ValueError('Probability vectors must have positive mass')
    # Softmax is computed in float32; renormalize after conversion to float64.
    values = values / mass
    if temperature == 1.0:
        return values
    logits = np.log(np.clip(values, np.finfo(np.float64).tiny, 1.0)) / temperature
    logits -= logits.max(axis=-1, keepdims=True)
    scaled = np.exp(logits)
    return scaled / scaled.sum(axis=-1, keepdims=True)

#### 5.6.3. Notebook prediction interface

Use the same logits, labels and review-policy semantics as the web application. This class performs inference only.


In [ ]:
class FashionClassifier:
    def __init__(self, checkpoint_path, device=None):
        checkpoint = checkpoint_path if isinstance(checkpoint_path, dict) else load_checkpoint(checkpoint_path)
        self.device = "/CPU:0" if device == "cpu" else (device if device and str(device).startswith("/") else None)
        self.target, self.labels = checkpoint["target"], list(checkpoint["labels"])
        self.temperature = float(checkpoint.get("temperature", 1.0))
        self.review_policy = checkpoint.get("review_policy")
        self.model_type = checkpoint["model_type"]
        self.model = checkpoint.get("model")
        self.members = None
        if self.model is not None:
            self.mean, self.std = checkpoint["mean"], checkpoint["std"]
            self.image_size = checkpoint["image_size"]
            self.estimator = self.feature_config = None
        else:
            raise ValueError("Classifier has no trained Keras model")

    def predict_batch(self, inputs):
        with tf.device(self.device):
            probabilities = tf.nn.softmax(self.model(inputs, training=False), axis=-1).numpy()
        return temperature_scale(probabilities, self.temperature)

    def predict_probabilities(self, image):
        inputs = image_batch(image, self.image_size, self.mean, self.std)
        return self.predict_batch(inputs)[0]

    def predict(self, image: Image.Image, top_k: int = 3) -> dict:
        if top_k < 1:
            raise ValueError("top_k must be positive")
        probabilities = self.predict_probabilities(image)
        count = min(top_k, len(self.labels))
        indices = np.argsort(probabilities)[::-1][:count]
        ranked = [
            {"label": self.labels[int(index)], "confidence": float(probabilities[index])}
            for index in indices
        ]
        result = {
            "target": self.target,
            "label": ranked[0]["label"],
            "confidence": ranked[0]["confidence"],
            "top_k": ranked,
        }
        if self.review_policy is not None:
            threshold = self.review_policy['threshold']
            result['needs_review'] = threshold is None or ranked[0]['confidence'] < threshold
            result['review_threshold'] = threshold
            result['confidence_calibrated'] = self.temperature != 1.0
            if self.review_policy.get('brightness_stability') and not result['needs_review']:
                stable_label = int(indices[0])
                unstable = any(
                    int(self.predict_probabilities(ImageEnhance.Brightness(image.convert('RGB')).enhance(factor)).argmax()) != stable_label
                    for factor in (0.8, 1.2)
                )
                if unstable:
                    result['needs_review'] = True
                    result['review_reason'] = 'Prediction changes with lighting'
        return result

#### 5.6.4. Evaluate a partition

Compute accuracy, supported-class macro-F1, ECE, NLL and Brier score. Record sampled batch/single-image differences as a diagnostic; large differences or changed predictions warn, while invalid probabilities raise an error.


In [ ]:
def evaluate_saved_checkpoint(checkpoint_path, frame, device="cpu", batch_size=64):
    """Re-evaluate a frozen artifact with the application's exact preprocessing.

    This never fits a model or changes its checkpoint. Batching changes only
    execution and may slightly affect floating-point scores; label ordering
    and saved temperature match the web API.
    """
    from sklearn.metrics import log_loss

    if frame.empty:
        raise ValueError("Evaluation requires at least one image")
    predictor = FashionClassifier(checkpoint_path, device=device)
    labels = predictor.labels
    truth = np.asarray([labels.index(label) for label in frame[predictor.target]])
    batches = []
    for start in range(0, len(frame), batch_size):
        inputs = []
        for path in frame.image_path.iloc[start:start + batch_size]:
            with Image.open(path) as source:
                inputs.append(image_batch(source, predictor.image_size, predictor.mean, predictor.std))
        batches.append(predictor.predict_batch(np.concatenate(inputs)))
    probabilities = np.vstack(batches)
    if (probabilities.shape != (len(frame), len(labels))
            or not np.isfinite(probabilities).all()
            or (probabilities < 0).any()
            or not np.allclose(probabilities.sum(axis=1), 1.0, atol=1e-5, rtol=1e-5)):
        raise ValueError("Batch inference returned invalid probabilities")
    predictions = probabilities.argmax(1)
    metrics = {
        "accuracy": float(accuracy_score(truth, predictions)),
        "macro_f1": supported_macro_f1(truth, predictions),
        "ece": expected_calibration_error(truth, probabilities),
        "nll": float(log_loss(truth, probabilities, labels=np.arange(len(labels)))),
        "brier": float(np.mean(np.sum((probabilities - np.eye(len(labels))[truth]) ** 2, axis=1))),
    }
    # Batch and single-image GPU kernels can produce slightly different scores.
    # Sample this difference as a diagnostic, not a requirement for calibration.
    # A 0.001 absolute difference means 0.1 percentage points of probability;
    # it is a warning threshold, not a guarantee of equivalent predictions.
    import warnings

    positions = sorted({0, len(frame) // 2, len(frame) - 1})
    max_absolute_difference = 0.0
    prediction_mismatches = []
    for position in positions:
        with Image.open(frame.image_path.iloc[position]) as source:
            single = np.asarray(predictor.predict_probabilities(source))
        if (single.shape != probabilities[position].shape
                or not np.isfinite(single).all()
                or (single < 0).any()
                or not np.isclose(single.sum(), 1.0, atol=1e-5, rtol=1e-5)):
            raise ValueError("Single-image inference returned invalid probabilities")
        difference = float(np.max(np.abs(probabilities[position] - single)))
        max_absolute_difference = max(max_absolute_difference, difference)
        if predictions[position] != single.argmax():
            prediction_mismatches.append(position)
    inference_consistency = {
        "sample_count": len(positions),
        "max_absolute_difference": max_absolute_difference,
        "prediction_mismatch_positions": prediction_mismatches,
    }
    if max_absolute_difference > 1e-3 or prediction_mismatches:
        warnings.warn(
            f"Batch/single-image inference diagnostic: {inference_consistency}. "
            "Evaluation and calibration use batched probabilities. Inspect these "
            "differences before interpreting single-image application results.",
            RuntimeWarning, stacklevel=2,
        )
    return {"labels": labels, "truth": truth, "predictions": predictions,
            "probabilities": probabilities, "metrics": metrics,
            "inference_consistency": inference_consistency}

#### 5.6.5. Fit temperature and the review threshold

Search temperature on calibration groups, then use policy groups to decide whether to keep it and when to request manual review.


In [ ]:
def calibrate(checkpoint, calibration, policy, device):
    checkpoint = dict(checkpoint)
    cal = evaluate_saved_checkpoint(checkpoint, calibration, device=str(device))
    raw = evaluate_saved_checkpoint(checkpoint, policy, device=str(device))
    fit = minimize_scalar(lambda log_t: log_loss(
        cal['truth'], temperature_scale(cal['probabilities'], np.exp(log_t)),
        labels=np.arange(len(checkpoint['labels']))),
        bounds=(np.log(0.25), np.log(10)), method='bounded')
    temperature = float(np.exp(fit.x))
    scaled = temperature_scale(raw['probabilities'], temperature)
    if (log_loss(raw['truth'], scaled, labels=np.arange(len(checkpoint['labels']))) < raw['metrics']['nll']
            and expected_calibration_error(raw['truth'], scaled) <= raw['metrics']['ece']):
        checkpoint['temperature'] = temperature
    else:
        scaled = raw['probabilities']
    threshold = None
    for value in np.round(np.arange(0.50, 1.00, 0.01), 2):
        accepted = scaled.max(1) >= value
        if accepted.sum() >= 100 and np.mean(scaled.argmax(1)[accepted] == raw['truth'][accepted]) >= 0.90:
            threshold = float(value)
            break
    checkpoint['review_policy'] = {'threshold': threshold, 'target_accuracy': 0.90,
                                   'minimum_policy_samples': 100,
                                   'brightness_stability': checkpoint['target'] == 'articleType'}
    return checkpoint

In [ ]:
method = winners['gender']
checkpoints['gender'] = dict(target='gender', labels=labels_by_target['gender'], model=models['gender'][method], model_type=method, temperature=1.0, mean=normalisation['mean'], std=normalisation['std'], image_size=list(IMAGE_SIZE), comparison_row=comparisons['gender'].loc[method].to_dict())
checkpoints['gender'] = calibrate(checkpoints['gender'], frames['gender']['calibration'], frames['gender']['policy'], DEVICE)
method = winners['usage']
checkpoints['usage'] = dict(target='usage', labels=labels_by_target['usage'], model=models['usage'][method], model_type=method, temperature=1.0, mean=normalisation['mean'], std=normalisation['std'], image_size=list(IMAGE_SIZE), comparison_row=comparisons['usage'].loc[method].to_dict())
checkpoints['usage'] = calibrate(checkpoints['usage'], frames['usage']['calibration'], frames['usage']['policy'], DEVICE)

#### 5.6.6. Evaluate the selected model on the internal test

Do not revise the architecture based on this table. Report prior development exposure and inspect per-class support before interpreting overall accuracy.


In [ ]:
test_results = {}
result = evaluate_saved_checkpoint(checkpoints['gender'], metadata_by_target['gender'].loc[metadata_by_target['gender']['split'].eq('test')], device=DEVICE)
test_results['gender'] = result
checkpoints['gender']['test_metrics'] = result['metrics']
display(pd.Series(result['metrics'], name='gender'))
report = classification_report(result['truth'], result['predictions'], labels=np.arange(len(result['labels'])), target_names=result['labels'], output_dict=True, zero_division=0)
display(pd.DataFrame(report).T)
pd.DataFrame(report).T.to_csv(RESULTS / f"{stem_for('gender')}_test_per_class.csv")
pd.Series(result['metrics']).to_csv(RESULTS / f"{stem_for('gender')}_test_metrics.csv")
result = evaluate_saved_checkpoint(checkpoints['usage'], metadata_by_target['usage'].loc[metadata_by_target['usage']['split'].eq('test')], device=DEVICE)
test_results['usage'] = result
checkpoints['usage']['test_metrics'] = result['metrics']
display(pd.Series(result['metrics'], name='usage'))
report = classification_report(result['truth'], result['predictions'], labels=np.arange(len(result['labels'])), target_names=result['labels'], output_dict=True, zero_division=0)
display(pd.DataFrame(report).T)
pd.DataFrame(report).T.to_csv(RESULTS / f"{stem_for('usage')}_test_per_class.csv")
pd.Series(result['metrics']).to_csv(RESULTS / f"{stem_for('usage')}_test_metrics.csv")

#### 5.6.7. Confidence bins

Compare average confidence with actual accuracy within each bin. Low-support bins are less reliable.


In [ ]:
def calibration_table(truth: np.ndarray, probabilities: np.ndarray) -> pd.DataFrame:
    """Return a ten-bin reliability table for notebook evidence."""
    confidence = probabilities.max(axis=1)
    correct = probabilities.argmax(axis=1) == np.asarray(truth)
    frame = pd.DataFrame({"confidence": confidence, "correct": correct})
    frame["bin"] = pd.cut(
        frame.confidence, bins=np.linspace(0, 1, 11), include_lowest=True
    )
    return frame.groupby("bin", observed=False).agg(
        mean_confidence=("confidence", "mean"),
        accuracy=("correct", "mean"),
        samples=("correct", "size"),
    )

In [ ]:
result = test_results['gender']
display(calibration_table(result['truth'], result['probabilities']))
result = test_results['usage']
display(calibration_table(result['truth'], result['probabilities']))

#### 5.6.8. Confusion matrices

Rows are true labels and columns are predicted labels. The article-type plot groups uncommon predicted classes into an Other column for readability.


In [ ]:
# Gender
from sklearn.metrics import confusion_matrix
import seaborn as sns
directory = RESULTS
stem = stem_for('gender')
result = test_results['gender']
labels = result['labels']
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay.from_predictions(result['truth'], result['predictions'], labels=np.arange(len(labels)), display_labels=labels, normalize='true', values_format='.2f', xticks_rotation=45, ax=ax, cmap='Blues', colorbar=False)
ax.set_title(f"{'gender'}: selected checkpoint / internal test")

In [ ]:
# Occasion (usage)
fig.tight_layout()
fig.savefig(FIGURES / f'{stem}_confusion.png', dpi=160, bbox_inches='tight')
plt.show()
directory = RESULTS
stem = stem_for('usage')
result = test_results['usage']
labels = result['labels']
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay.from_predictions(result['truth'], result['predictions'], labels=np.arange(len(labels)), display_labels=labels, normalize='true', values_format='.2f', xticks_rotation=45, ax=ax, cmap='Blues', colorbar=False)
ax.set_title(f"{'usage'}: selected checkpoint / internal test")
fig.tight_layout()
fig.savefig(FIGURES / f'{stem}_confusion.png', dpi=160, bbox_inches='tight')
plt.show()

## 6. Ultimate Judgement

**Recorded selected-model performance.**

| Target | Selected model | Validation accuracy | Validation macro-F1 | Internal-test accuracy | Internal-test macro-F1 | ECE |
|---|---|---:|---:|---:|---:|---:|
| gender | cnn_four_blocks_scheduled | 89.86% | 0.7909 | 88.61% | 0.7396 | 0.0114 |
| usage | shallow_mlp_lower_lr | 87.38% | 0.4522 | 85.69% | 0.3745 | 0.0231 |

Models are selected by validation macro-F1. Internal-test scores have prior development exposure and are descriptive, not an independent model-selection criterion.


In [ ]:
winner = comparisons['gender'].loc[winners['gender']]
baseline = comparisons['gender'].loc[comparisons['gender']['family'].eq('shallow_mlp')].iloc[0]
display(pd.Series({'selected_method': winners['gender'], 'selection_macro_f1': winner.validation_macro_f1, 'selection_accuracy': winner.validation_accuracy, 'macro_f1_gain_over_baseline': winner.validation_macro_f1 - baseline.validation_macro_f1, 'internal_test_accuracy': test_results['gender']['metrics']['accuracy'], 'internal_test_macro_f1': test_results['gender']['metrics']['macro_f1']}, name='gender'))
winner = comparisons['usage'].loc[winners['usage']]
baseline = comparisons['usage'].loc[comparisons['usage']['family'].eq('shallow_mlp')].iloc[0]
display(pd.Series({'selected_method': winners['usage'], 'selection_macro_f1': winner.validation_macro_f1, 'selection_accuracy': winner.validation_accuracy, 'macro_f1_gain_over_baseline': winner.validation_macro_f1 - baseline.validation_macro_f1, 'internal_test_accuracy': test_results['usage']['metrics']['accuracy'], 'internal_test_macro_f1': test_results['usage']['metrics']['macro_f1']}, name='usage'))

### 6.1. Decision Analysis

For gender, the 256-unit shallow MLP reaches 0.7118 validation macro-F1; the two-layer deeper MLP (256, 128) reaches 0.6912. Two hidden layers improved the deeper-family result relative to the previously measured three-layer choice (0.6409). The four-block CNN remains strongest at 0.7909 macro-F1 and 89.86% validation accuracy, with 654,021 parameters. The architecture is unchanged; a small difference between training runs is not evidence of an architectural gain. Its internal-test accuracy is 88.61% and macro-F1 is 0.7396. Predictions describe catalogue audience labels, not personal identity.

For occasion, the 256-unit shallow MLP wins narrowly at 0.4522 validation macro-F1, against 0.4497 for weighted CNN continuation and 0.3694 for the deeper MLP. The deeper family retains three hidden layers (256, 128, 64) with Adam 0.001 because that measured configuration exceeded the tested alternatives. This differs from gender because configuration selection follows each target's validation evidence, not a requirement to use different architectures.

The occasion CNN has higher validation accuracy (89.55% versus 87.38%) and fewer parameters, while the MLP wins the declared macro-F1 criterion. Its small advantage depends on very limited rare-class support and should be treated as provisional. The selected MLP records 85.69% internal-test accuracy and 0.3745 macro-F1. Neither a high overall accuracy nor a small single-seed F1 lead establishes broad minority-class reliability. Calibration and review policies do not remove annotation ambiguity or the internal test's prior development exposure.


## 7. Final Prediction

Save the selected model, reload it, and predict a sample image. Exported metadata allows the standalone prediction scripts and web application to apply the same preprocessing and confidence policy.


In [ ]:
def save_checkpoint(checkpoint, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path = path.with_suffix(".keras")
    model = checkpoint["model"]
    metadata = {key: value for key, value in checkpoint.items() if key != "model"}
    metadata = json.loads(json.dumps(metadata, default=lambda value: value.item()))
    model.get_layer("metadata").metadata = metadata
    # Export architecture, learned weights and metadata without optimizer slots.
    inference_model = type(model).from_config(model.get_config())
    inference_model.set_weights(model.get_weights())
    inference_model.save(path)
    return path

In [ ]:
stem, method = (stem_for('gender'), winners['gender'])
path = save_checkpoint(checkpoints['gender'], OUTPUT / f'{stem}_model.keras')
histories['gender'][method].to_csv(RESULTS / f'{stem}_history.csv', index=False)
summary = dict(target='gender', selected=method, cnn_selected=ranked(tuning['gender']).index[0], framework='tensorflow_keras', model_file=path.name, test_metrics=test_results['gender']['metrics'], split_sizes={name: len(frame) for name, frame in frames['gender'].items()}, test_scope='internal_test_with_prior_development_exposure')
(RESULTS / f'{stem}_summary.json').write_text(json.dumps(summary, indent=2) + '\n')
print('Saved', path)
stem, method = (stem_for('usage'), winners['usage'])
path = save_checkpoint(checkpoints['usage'], OUTPUT / f'{stem}_model.keras')
histories['usage'][method].to_csv(RESULTS / f'{stem}_history.csv', index=False)
summary = dict(target='usage', selected=method, cnn_selected=ranked(tuning['usage']).index[0], framework='tensorflow_keras', model_file=path.name, test_metrics=test_results['usage']['metrics'], split_sizes={name: len(frame) for name, frame in frames['usage'].items()}, test_scope='internal_test_with_prior_development_exposure')
(RESULTS / f'{stem}_summary.json').write_text(json.dumps(summary, indent=2) + '\n')
print('Saved', path)

### 7.1. Load the Saved Model & Predict

A successful reload checks the saved architecture and metadata. This example is a functional check, not an independent quality estimate.


In [ ]:
def resolve_classifier_path(path):
    path = Path(path)
    if path.exists():
        return path
    raise FileNotFoundError(f"Missing trained classifier: {path}. Run the classification notebook first.")

def load_checkpoint(path):
    path = resolve_classifier_path(path)
    if path.suffix == ".keras":
        model = keras.models.load_model(path, compile=False)
        return {**dict(model.get_layer("metadata").metadata), "model": model}
    raise ValueError(f"Unsupported classifier format: {path.suffix}")

In [ ]:
predictor = FashionClassifier(OUTPUT / f"{stem_for('gender')}_model.keras")
with Image.open(frames['gender']['selection'].image_path.iloc[0]) as image:
    print(predictor.predict(image))
predictor = FashionClassifier(OUTPUT / f"{stem_for('usage')}_model.keras")
with Image.open(frames['usage']['selection'].image_path.iloc[0]) as image:
    print(predictor.predict(image))

## 8. Conclusion

For gender, the 256-unit shallow MLP reaches 0.7118 validation macro-F1; the two-layer deeper MLP (256, 128) reaches 0.6912. Two hidden layers improved the deeper-family result relative to the previously measured three-layer choice (0.6409). The four-block CNN remains strongest at 0.7909 macro-F1 and 89.86% validation accuracy, with 654,021 parameters. The architecture is unchanged; a small difference between training runs is not evidence of an architectural gain. Its internal-test accuracy is 88.61% and macro-F1 is 0.7396. Predictions describe catalogue audience labels, not personal identity.

For occasion, the 256-unit shallow MLP wins narrowly at 0.4522 validation macro-F1, against 0.4497 for weighted CNN continuation and 0.3694 for the deeper MLP. The deeper family retains three hidden layers (256, 128, 64) with Adam 0.001 because that measured configuration exceeded the tested alternatives. This differs from gender because configuration selection follows each target's validation evidence, not a requirement to use different architectures.

The occasion CNN has higher validation accuracy (89.55% versus 87.38%) and fewer parameters, while the MLP wins the declared macro-F1 criterion. Its small advantage depends on very limited rare-class support and should be treated as provisional. The selected MLP records 85.69% internal-test accuracy and 0.3745 macro-F1. Neither a high overall accuracy nor a small single-seed F1 lead establishes broad minority-class reliability. Calibration and review policies do not remove annotation ambiguity or the internal test's prior development exposure.
